# Composite Entailment

In [ ]:
import torch

from sklearn.metrics import roc_auc_score
from subspaces import *
from train_nli import NLITrainingData
from train_nli_baseline import NLITrainingData as NLITrainingDataBaseline
from model import *

device = "cuda"

# Load premise & composite hypotheses

In [ ]:
premises = []
hypotheses_a = []
hypotheses_b = []
labels = []

with open("./composite_entailment.txt", "r", encoding="utf-8") as f:
    next(f)
    for line in f:
        line = line.strip()
        if not line:
            continue  
        parts = line.split(".")
        premise = parts[0]
        hypothesis_a = parts[1]
        hypothesis_b = ",".join(parts[2:-1])
        label = int(parts[-1])
        premises.append(premise)
        hypotheses_a.append(hypothesis_a)
        hypotheses_b.append(hypothesis_b)
        labels.append(label)
        
labels = (torch.tensor(labels, device=device) == 0).long() # 1: Entailment; 0: Non-entailment

flipped_labels = True # negate hypothesis (b)
if flipped_labels:
    labels = 1 - labels
print("Loaded", len(premises), "rows")

## Subspace Embeddings (SE)

In [ ]:
#config = NLITrainingData.load("./nli_models/all-MiniLM-L6-v2_64x64_lbd0.05_context35_seed13_2way/config.pt")
#config = NLITrainingData.load("./nli_models/all-MiniLM-L6-v2_128x128_lbd0.05_context35_seed13_2way/config.pt")
#config = NLITrainingData.load("./nli_models/all-mpnet-base-v2_64x64_lbd0.05_context35_seed24_2way/config.pt")
config = NLITrainingData.load("./nli_models/all-mpnet-base-v2_128x128_lbd0.05_context35_seed24_2way/config.pt")

model = TransformerSubspaceEmbedder(
    config.base_model_name, config.N, config.D, config.lbd, two_way=config.two_way, cache_dir="./.cache"
);
model.load_state_dict(config.state_dict, strict=False);
model.eval();
model.to(device);

In [ ]:
batch_size = 4
proj_p  = [] # Premise
proj_ha = [] # Hypothesis (a)
proj_hb = [] # Hypothesis (b)
for i in tqdm(range(0, len(premises), batch_size)):
    proj_p_batch  = model.encode(premises[i:i+batch_size], max_length=config.max_length, device=device)
    proj_ha_batch = model.encode(hypotheses_a[i:i+batch_size], max_length=config.max_length, device=device)
    proj_hb_batch = model.encode(hypotheses_b[i:i+batch_size], max_length=config.max_length, device=device)
    proj_p.append(proj_p_batch)
    proj_ha.append(proj_ha_batch)
    proj_hb.append(proj_hb_batch)
proj_p = torch.cat(proj_p)
proj_ha = torch.cat(proj_ha)
proj_hb = torch.cat(proj_hb)

I = torch.eye(config.D, device=device)[None].expand((len(proj_hb), config.D, config.D))
if flipped_labels:
    proj_hb = I - proj_hb

In [ ]:
proj_composite = proj_hb @ proj_ha @ proj_hb
u, s, vh = torch.linalg.svd(proj_composite)
proj_composite = (u * (s.view(-1,1,config.D) > 0.1)) @ vh

prior = torch.einsum("bii->b", proj_p)

scores_p_hb = torch.einsum("bii->b", (torch.bmm(proj_p, proj_hb))) / prior
scores_p_composite = torch.einsum("bii->b", (torch.bmm(proj_p, proj_composite))) / prior

p_hb_auc = roc_auc_score(labels.detach().cpu().numpy(), scores_p_hb.detach().cpu().numpy())
p_composite_auc = roc_auc_score(labels.detach().cpu().numpy(), scores_p_composite.detach().cpu().numpy())

print(f"ROC AUC premise - hypothesis (b): {p_hb_auc * 100:.2f}")
print(f"ROC AUC premise - hypothesis (a) AND {'NOT ' if flipped_labels else ''}hypothesis (b): {p_composite_auc * 100:.2f}")

## Euclidean Vector Embeddings

In [ ]:
#model_name = "./nli_models/baseline_all-MiniLM-L6-v2_context35_seed8_2way/config.pt"
#model_name = "./nli_models/baseline_p-h_all-MiniLM-L6-v2_context35_seed13_2way/config.pt"
#model_name = "./nli_models/baseline_all-mpnet-base-v2_context35_seed13_2way/config.pt"
model_name = "./nli_models/baseline_p-h_all-mpnet-base-v2_context35_seed13_2way/config.pt"

config = NLITrainingDataBaseline.load(model_name)

model = TransformerClassifier(
    config.base_model_name, difference="p-h" in model_name, two_way=config.two_way, cache_dir="./.cache"
);
model.load_state_dict(config.state_dict, strict=False);
model.eval();
model.to(device);

In [ ]:
batch_size = 4
emb_p = [] # Premise
emb_ha = [] # Hypothesis (a)
emb_hb = [] # Hypothesis (b)

for i in tqdm(range(0, len(premises), batch_size)):
    emb_p_batch = model.encode(premises[i:i+batch_size], max_length=config.max_length, device=device)
    emb_ha_batch = model.encode(hypotheses_a[i:i+batch_size], max_length=config.max_length, device=device)
    emb_hb_batch = model.encode(hypotheses_b[i:i+batch_size], max_length=config.max_length, device=device)
    emb_p.append(emb_p_batch)
    emb_ha.append(emb_ha_batch)
    emb_hb.append(emb_hb_batch)
emb_p = torch.cat(emb_p)
emb_ha = torch.cat(emb_ha)
emb_hb = torch.cat(emb_hb)

In [ ]:
if flipped_labels:
    emb_c = emb_ha - emb_hb
else:
    emb_c = 0.5 * emb_ha + 0.5 * emb_hb

entailment_score_p_hb = F.softmax(model.classify(emb_p, emb_hb), dim=-1)[:,0]
entailment_score_p_composite = F.softmax(model.classify(emb_p, emb_c), dim=-1)[:,0]

if flipped_labels:
    p_hb_auc = roc_auc_score(1 - labels.detach().cpu().numpy(), entailment_score_p_hb.detach().cpu().numpy())
else:
    p_hb_auc = roc_auc_score(labels.detach().cpu().numpy(), entailment_score_p_hb.detach().cpu().numpy())
    
p_composite_auc = roc_auc_score(labels.detach().cpu().numpy(), entailment_score_p_composite.detach().cpu().numpy())

print(f"ROC AUC premise - hypothesis (b): {p_hb_auc * 100:.2f}")
print(f"ROC AUC premise - hypothesis (a) AND {'NOT ' if flipped_labels else ''}hypothesis (b): {p_composite_auc * 100:.2f}")